# Procédure d'obfuscation AloePri — Qwen/Qwen3-8B

**Objectif** — Reproduire de bout en bout la procédure d'obfuscation AloePri
sur `Qwen/Qwen3-8B` :
1. **Section 1** — construction des matrices d'obfuscation (permutation de
   vocabulaire Π, bruit d'embedding, facteurs d'attention/FFN, matrices clés
   P̂/Q̂) ;
2. **Section 2** — transformation du modèle sur Modal (streaming, ~16 Go),
   vérification bit-à-bit et récupération des clés ;
3. **Section 3** — export et service serverless du modèle obfusqué.

Les sections 4-6 (attaque ISA, évaluation, bilan) sont ajoutées par des
tâches ultérieures : ce notebook est le socle exécutable (sections 0-3).

**Modèle de menace** — L'opérateur du serveur d'inférence ne doit pas pouvoir
reconstruire la représentation interne du modèle à partir des seuls poids
obfusqués (permutation de vocabulaire, bruit α_e/α_h, facteurs orthogonaux,
matrices clés). Les clés restent **exclusivement côté client** : elles ne sont
jamais montées par le service.

**Références** — Papier : [AloePri — arXiv:2603.01499](https://arxiv.org/pdf/2603.01499)
(Algorithme 1, §5.2.2, §5.4) · Spec : `docs/superpowers/specs/2026-08-24-aloepri-notebook-design.md`
· Code : package `aloepri/` (port T1-T3, 40/40 tests) + `modal_app.py`.

---

## 0. Setup

In [ ]:
import os, sys
# bootstrap : racine du worktree sur sys.path. Le kernel nbconvert démarre
# dans le dossier du notebook (notebooks/) ; on remonte jusqu'à trouver le
# package aloepri/ pour que les imports des cellules 1.5 et 2.1 fonctionnent
# quel que soit le point de lancement (nbconvert, Jupyter racine ou notebooks/).
for _ in range(6):
    if os.path.isdir(os.path.join(os.getcwd(), "aloepri")):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import torch

# Constantes de la procédure (valeurs du plan — à utiliser telles quelles)
MODEL = "Qwen/Qwen3-8B"
SEED = 0
ALPHA_E = 0.3
BETA = 8
THINK_ID = 151667

# Drapeau : False → exécution locale rapide (sections 0-1 + branches "sauté") ;
# True → exécute réellement les cellules Modal (transform, verify, deploy,
# health check). Passer à True uniquement avec une CLI Modal authentifiée.
RUN_HEAVY = False

In [ ]:
import os

print(f"racine      {os.getcwd()}")
print(f"torch      {torch.__version__}")
print(f"numpy      {np.__version__}")
assert isinstance(RUN_HEAVY, bool), "RUN_HEAVY doit être un booléen"
print(f"RUN_HEAVY  {RUN_HEAVY} (booléen ✓)")

modal_cli = os.path.expanduser("~/modal-venv/bin/modal")
print(f"Modal CLI  {modal_cli} -> " + ("accessible ✓" if os.path.exists(modal_cli)
                                       else "introuvable (cellules RUN_HEAVY indisponibles)"))

### Comment exécuter ce notebook

- **Environnement local** — venv `.venv/` (numpy 2.5, torch 2.13 CPU,
  transformers 5.15, jupyter 1.1.1). Lancer Jupyter **depuis la racine du
  worktree** (`jupyter notebook notebooks/aloepri_procedure.ipynb`) : les
  imports `aloepri` (cellules 1.5 et 2.1) en dépendent.
- **`RUN_HEAVY = False` (défaut)** — exécution locale rapide : sections 0-1
  complètes ; les cellules Modal des sections 2-3 affichent
  `[RUN_HEAVY=False] … sauté` (aucun appel Modal, aucune authentification).
- **`RUN_HEAVY = True`** — exécute réellement les cellules Modal : transform
  (~16 Go, ~30-60 min CPU), récupération + suppression des clés, verify
  (~30 min), deploy + health check. Prérequis : CLI `~/modal-venv/bin/modal`
  authentifiée (`~/modal-venv/bin/modal token new`) ; les volumes Modal
  (`obfuscator-models`, `obfuscator-keys`) sont créés automatiquement.
- **Coûts** — transform ≈ 1 h CPU ; verify ≈ 30 min CPU ; serve : GPU L4
  scale-to-zero (facturation à la seconde). Les clés sont téléchargées dans
  `artifacts/obfuscation_keys.json` (gitignoré).

## 1. Matrices d'obfuscation

Cellules autonomes (numpy/torch, petites échelles) qui illustrent chaque
facteur de la procédure ; la transformation réelle (section 2) applique les
mêmes constructions aux tenseurs du modèle.

### 1.1 Permutation de vocabulaire Π

Permutation déterministe (seed `SEED`) des `V` lignes d'embedding / colonnes
de `lm_head`. `unperm` est l'inverse exacte : Π·Π⁻¹ = Id.

In [ ]:
import numpy as np
V = 1000
rng = np.random.default_rng(SEED)
perm = rng.permutation(V)
unperm = np.empty_like(perm); unperm[perm] = np.arange(V)
# vérification : Π·Π⁻¹ = Id
assert (perm[unperm] == np.arange(V)).all() and (unperm[perm] == np.arange(V)).all()
print(f"Π construite : {V} tokens, inverse exacte ✓")

### 1.2 Bruit d'embedding (α_e, α_h)

Bruit gaussien relatif aux poids : σ(bruit) = α_e · σ(W) (embedding/head) ;
le même principe s'applique aux activations cachées avec α_h.

In [ ]:
w = torch.randn(64, 128)
noise = ALPHA_E * torch.randn_like(w) * w.std()   # rapport bruit/signal = α_e
assert abs(noise.std() / w.std() - ALPHA_E) < 0.1
print(f"σ(bruit)/σ(poids) ≈ {noise.std()/w.std():.2f} ≈ α_e ✓")

### 1.3 Facteurs d'attention

- **R̂** — rotation RoPE par paires `(i, i+d/2)` (layout `half` de
  `rotate_half`, Qwen2/Qwen3) ;
- **Ĥ** — facteur diagonal — **désactivé sur Qwen3** (`rope_scaling=off`) :
  les normes de tête `q_norm`/`k_norm` ne commutent pas avec un facteur
  diagonal ;
- **Ẑ** — permutation de blocs de largeur `d_head/β` ;
- **Û_vo** — matrice orthogonale (décomposition QR).

In [ ]:
d_head = 32
beta = BETA  # β du papier (8) ; l'exemple ci-dessous illustre β_ex = 3 blocs
# Ẑ : permutation de blocs de largeur d_head//3 (exemple β_ex=3). d_head=32
# n'est pas multiple de 3 → les lignes restantes sont laissées à l'identité
# pour que Ẑ soit une permutation complète (orthogonale).
blk = [1, 2, 0]
w = d_head // len(blk)
Z = torch.zeros(d_head, d_head)
for j, src in enumerate(blk):
    Z[j*w:(j+1)*w, src*w:(src+1)*w] = torch.eye(w)
for r in range(len(blk) * w, d_head):
    Z[r, r] = 1.0
assert torch.allclose(Z @ Z.T, torch.eye(d_head)), "Ẑ doit être une permutation (orthogonale)"
# Û_vo : orthogonale via QR
U, _ = torch.linalg.qr(torch.randn(d_head, d_head))
assert torch.allclose(U.T @ U, torch.eye(d_head), atol=1e-5), "Û_vo doit être orthogonale"
print("R̂/Ẑ/Û_vo : facteurs orthogonaux vérifiés ✓")

### 1.4 Facteurs FFN

Permutation des neurones de la couche intermédiaire + scaling multiplicatif
`exp(N(0, 0.1))` (bruit relatif ~10 % par neurone).

In [ ]:
h = 64
rng = np.random.default_rng(SEED + 7)
neu = rng.permutation(h)
scale = torch.exp(0.1 * torch.randn(h))
assert len(set(neu.tolist())) == h
print(f"FFN : permutation de {h} neurones + scalings ∈ [exp(±0.1·N)] ✓")

### 1.5 Matrices clés P̂/Q̂ — aperçu (Algorithme 1)

Implémentation complète à venir en **Section 6** ; démo de l'API
`aloepri.key_matrix` (Algorithme 1, arXiv:2603.01499 §5.4) : `P̂` de forme
`(d, d+2h)` et `Q̂` de forme `(d+2h, d)` tels que **P̂·Q̂ = I_d**.

In [ ]:
from aloepri.key_matrix import init_key_matrix, key_mat_gen, inv_key_mat_gen
import numpy.random as npr
base = init_key_matrix(d=64, h=8, lam=0.3, rng=npr.default_rng(SEED))
P = key_mat_gen(base); Q = inv_key_mat_gen(base)
err = float(np.abs(P @ Q - np.eye(64)).max())
assert err < 1e-10, f"P̂·Q̂=I attendu, erreur max {err}"
print(f"P̂ ({P.shape}) · Q̂ ({Q.shape}) = I, erreur max {err:.2e} ✓")

## 2. Obfuscation du modèle (Modal)

### 2.1 Vérification d'architecture

`aloepri.check_arch` valide les hypothèses du POC sur `Qwen/Qwen3-8B`
(config seule — aucun poids téléchargé) : GQA (`num_key_value_heads` divise
`num_attention_heads`), `head_dim` pair (RoPE), biais q/k/v, **`q_norm` /
`k_norm` (Qwen3) → `rope_scaling=off`**, layout RoPE `half` (rotate_half),
weight tying, cohérence du vocabulaire.

In [ ]:
from aloepri.check_arch import check  # API réelle : check() (le brief disait check_arch)
import contextlib, io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    rc = check(MODEL)  # télécharge uniquement la config, jamais les poids
print(buf.getvalue(), end="")
assert rc == 0, "check() a signalé des hypothèses en échec"
assert "toutes les hypothèses sont satisfaites" in buf.getvalue(), \
    "le rapport doit conclure : toutes les hypothèses sont satisfaites"
print("✓ toutes les hypothèses sont satisfaites — la transformation peut être lancée")

### 2.2 Transformation sur Modal

Transformation réelle en streaming (mémoire-léger, ~16 Go de poids, ~30-60
min CPU) : écrit le modèle obfusqué sur le volume `obfuscator-models`
(`qwen3-8b-obf`) et les clés sur `obfuscator-keys`. Cellule conditionnée par
`RUN_HEAVY` — appel CLI robuste (équivalent de
`modal.Function.lookup("obfuscator-aloepri","transform").remote(seed=SEED,
alpha_e=ALPHA_E, beta=BETA)`) :

```bash
~/modal-venv/bin/modal run modal_app.py::transform --seed 0 --alpha-e 0.3 --beta 8
```

In [ ]:
if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::transform --seed 0 --alpha-e 0.3 --beta 8
else:
    print("[RUN_HEAVY=False] transform() sauté — résultat attendu : keys_sha256 + out_subdir='qwen3-8b-obf'")

### 2.3 Vérification bit-à-bit + récupération des clés

1. Télécharger les clés du volume `obfuscator-keys` vers
   `artifacts/obfuscation_keys.json` (gitignoré) ;
2. **Supprimer le volume clés** — posture : les clés ne restent jamais sur
   Modal (`verify()` régénère les clés par seed, il n'en a pas besoin) ;
3. `verify()` avec **les mêmes hyperparamètres que `transform()`**
   (`--seed 0 --alpha-e 0.3 --beta 8`) — échantillons de lignes embed/head +
   couches complètes ; assert : 0 écart bit-à-bit.

In [ ]:
if RUN_HEAVY:
    import os
    os.makedirs("artifacts", exist_ok=True)
    !~/modal-venv/bin/modal volume get obfuscator-keys /obfuscation_keys.json artifacts/obfuscation_keys.json
    # Posture de sécurité : les clés restent côté client (artifacts/) — le
    # volume clés est supprimé de Modal (verify() régénère les clés par seed).
    !~/modal-venv/bin/modal volume delete obfuscator-keys -y
    verify_out = !~/modal-venv/bin/modal run modal_app.py::verify --seed 0 --alpha-e 0.3 --beta 8
    print(verify_out.s)
    assert "[OK]" in verify_out.s, "verify() doit rapporter 0 écart bit-à-bit sur les échantillons"
    print("✓ verify() : 0 écart bit-à-bit sur les échantillons")
else:
    print("[RUN_HEAVY=False] récupération des clés + verify() sauté — attendu : 0 écart bit-à-bit")

## 3. Export et service (Modal)

### 3.1 Volume modèle

Le modèle obfusqué est servi depuis le volume `obfuscator-models`,
sous-répertoire `/qwen3-8b-obf` (écrit par `transform()`, section 2.2).
Inspection manuelle (authentification Modal requise) :
`~/modal-venv/bin/modal volume ls obfuscator-models /qwen3-8b-obf`.

In [ ]:
if RUN_HEAVY:
    !~/modal-venv/bin/modal deploy modal_app.py
else:
    print("[RUN_HEAVY=False] deploy sauté — attendu : https://mauceri--obfuscator-aloepri-serve.modal.run")

### 3.2 Cold start

Le service est scale-to-zero : le premier appel après une période d'inactivité
peut renvoyer `503` (boot du conteneur, ~1-2 min). La boucle de health check
ci-dessous attend jusqu'à 5 min avant d'abandonner.

In [ ]:
URL = "https://mauceri--obfuscator-aloepri-serve.modal.run"

if RUN_HEAVY:
    import os, time, requests
    api_key = open(os.path.expanduser("~/.aloepri-api-key")).read().strip() \
        if os.path.exists(os.path.expanduser("~/.aloepri-api-key")) else None
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    for _ in range(60):
        try:
            if requests.get(f"{URL}/health", headers=headers, timeout=10).status_code == 200:
                print("service prêt ✓"); break
        except requests.RequestException:
            time.sleep(5)
    else:
        raise SystemExit("service injoignable après 5 min")
else:
    print("[RUN_HEAVY=False] health check sauté — attendu : service prêt ✓")

## 4. Tests de base

### 4.1 Codec client (permute / dépermute)

La **permutation Π** est le secret : le client permute les ids des tokens
avant l'envoi — le serveur ne voit que des **nombres** (aucun tokenizer,
aucune clé sur Modal) et renvoie des ids permutés ; le client dépermute avec
Π⁻¹ (clés locales `artifacts/obfuscation_keys.json`, téléchargées en 2.3 puis
supprimées du volume Modal). Le codec inline ci-dessous est **réutilisé par la
section 5** (dépermutation des ids récupérés par l'attaque).

> Les clés n'existent localement qu'après un run lourd (section 2.3) : si le
> fichier est absent, la cellule affiche un message et s'arrête (branche
> `else`) — le codec n'est alors pas défini, et 4.2 / la section 5 ne
> s'exécutent qu'avec `RUN_HEAVY=True`.

In [ ]:
import json, os
from transformers import AutoTokenizer

if os.path.exists("artifacts/obfuscation_keys.json"):
    keys = json.load(open("artifacts/obfuscation_keys.json"))
    tok = AutoTokenizer.from_pretrained(MODEL)
    perm = {int(k): int(v) for k, v in keys["vocab_permutation"].items()}
    unperm = {int(k): int(v) for k, v in keys["vocab_unpermute"].items()}

    def encode(text):
        return [perm[i] for i in tok.encode(text)]

    def decode(ids):
        return tok.decode([unperm[i] for i in ids])

    # round-trip : perm puis déperm = identité
    clear = tok.encode("Quelle est la capitale de la France ?")
    assert decode(encode("Quelle est la capitale de la France ?")) == tok.decode(clear)
    print(f"codec ✓ — prompt clair : {len(clear)} tokens → {len(encode('x'))} ids permutés")
else:
    print("[clés absentes — exécuter la section 2 (RUN_HEAVY=True) d'abord]")

### 4.2 Questions simples

Décodage validé de bout en bout sur le service déployé (section 3) : template
**non-thinking** (`enable_thinking=False` — le template Qwen3 ferme alors le
bloc `<think>`), greedy (`do_sample=False`), `repetition_penalty=1.05`,
**blocage du token `<think>`** (`bad_words_ids=[[perm[THINK_ID]]]`, THINK_ID
= 151667 dans l'espace permuté). Pour chaque prompt : template → ids clairs →
`encode()` → `POST {URL}/generate` → `decode()` des ids générés — réponse
non vide et sans token `<think>`.

In [ ]:
if RUN_HEAVY:
    import requests

    PROMPTS = [
        "Quelle est la capitale de la France ?",
        "What is 17 times 23 ?",
        "Write a haiku about the sea.",
    ]
    for prompt in PROMPTS:
        # template non-thinking (Qwen3) : `enable_thinking=False` insère le
        # marqueur de fin <think> — le modèle répond directement. Le token
        # <think> (151667) est en plus bloqué en génération via bad_words_ids
        # (dans l'espace permuté).
        template = tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        perm_ids = encode(template)
        r = requests.post(
            f"{URL}/generate",
            json={
                "input_ids": perm_ids,
                "max_new_tokens": 120,
                "repetition_penalty": 1.05,
                "bad_words_ids": [[perm[THINK_ID]]],
            },
            timeout=300,
        )
        r.raise_for_status()
        # le serveur renvoie la séquence complète (entrée + généré) : on
        # retire les ids d'entrée avant de dépermuter/décoder
        answer = decode(r.json()["output_ids"][len(perm_ids):])
        print(f"--- {prompt}\n{answer}")
        assert answer.strip(), f"réponse vide pour {prompt!r}"
        assert "<think>" not in answer, f"token <think> dans la réponse pour {prompt!r}"
    print("✓ 4.2 : décodage validé — 3 réponses non vides, sans <think>")
else:
    print("[RUN_HEAVY=False] questions simples sauté — attendu : 3 réponses décodées (greedy, sans <think>)")

## 5. Attaques ISA — arc complet

### 5.1 Méthode ISA

Attaque par inversion d'état interne (Internal State Attack, arXiv 2507.16372 ;
AloePri Appendix D.1) — **l'attaquant = l'opérateur du serveur** : il possède
les poids (obfusqués) et observe l'inférence d'UNE requête réelle, mais
**pas la clé** de permutation.

1. **Capture** : état interne du prompt secret (ids permutés) — pondérations
   d'attention de la couche 0 (`channel="attn"`) ou état caché d'une couche
   (`channel="hidden"`) ;
2. **Candidat** : logits par position `P ∈ R^{T×vocab}` → entrée du modèle
   `embeds = softmax(P/τ) @ W_embed` (différentiable : le gradient remonte
   jusqu'à P à travers la table d'embedding) ;
3. **Phase 1** : Adam + **recuit de température** τ (3 → 0,1) sur une perte
   **relative** (MSE / variance de la cible — les états d'un vrai LLM ont des
   amplitudes énormes, une MSE brute écraserait le gradient) ;
4. **Phase 2** : ré-initialisation des logits près de l'argmax de la phase 1 +
   optimisation à température basse (corrige les choix discrets figés par le
   recuit) ;
5. **Résultat** : `ids_récupérés = argmax(P)` ; métrique = taux de
   correspondance avec les ids réellement envoyés au modèle
   (`taux_recuperation_ids_modele`). Sur le modèle obfusqué, ces ids sont
   **permutés** : les retrouver ne donne pas le texte sans la clé.

### 5.2 Arc des variantes

Variantes miniatures (`Qwen/Qwen3-0.6B`, seed 0, α_e=0.3, β=8) sur le volume
`obfuscator-models`, attaque canal `attn` (couche 0, 300 steps) :

| Variante | attn | perm | Résultat attendu |
|---|---|---|---|
| partiel (sans attn, **avec** perm) | claire | oui | ids **permutés** récupérés — texte illisible sans clé |
| baseline (Qwen3-0.6B clair) | claire | non | fuite **en clair** |
| partiel (sans attn, sans perm) | claire | non | fuite **en clair** (canal attn) |
| complet | obfusquée | oui | défendu (taux faible) |

Présentation dans la **priorité utilisateur** : la variante « sans attention,
AVEC permutation » d'abord (elle démontre que la permutation protège le texte
même sans défense d'attention — les ids récupérés sont permutés), puis les
fuites (baseline, partiel sans perm → le texte clair fuit via les scores
d'attention), puis le modèle complet (défendu).

In [ ]:
if RUN_HEAVY:
    # Variantes miniatures sur le volume — mêmes hyperparamètres que le
    # transform canonique : seed 0, alpha_e=0.3, beta=8 (seed identique
    # partout). Syntaxe booléenne de la CLI Modal 1.5.2 : drapeaux par paire
    # `--obfuscate-attention` / `--no-obfuscate-attention` (idem
    # `--apply-permutation`) — `--obfuscate-attention false` serait rejeté
    # par click (drapeau, pas de valeur).
    !~/modal-venv/bin/modal run modal_app.py::transform --model-name Qwen/Qwen3-0.6B --out-subdir qwen3-0.6b-partial-noperm --no-obfuscate-attention --no-apply-permutation --seed 0 --alpha-e 0.3 --beta 8
    !~/modal-venv/bin/modal run modal_app.py::transform --model-name Qwen/Qwen3-0.6B --out-subdir qwen3-0.6b-partial-perm --no-obfuscate-attention --apply-permutation --seed 0 --alpha-e 0.3 --beta 8
    !~/modal-venv/bin/modal run modal_app.py::transform --model-name Qwen/Qwen3-0.6B --out-subdir qwen3-0.6b-obf --obfuscate-attention --apply-permutation --seed 0 --alpha-e 0.3 --beta 8
    print("✓ 5.3 : variantes produites — qwen3-0.6b-partial-noperm / -partial-perm / -obf")
else:
    print("[RUN_HEAVY=False] production des variantes sauté — attendu : 3 sous-répertoires qwen3-0.6b-* sur obfuscator-models")

In [ ]:
if RUN_HEAVY:
    import json as _json
    import re as _re

    SECRET_PROMPT = "Quelle est la capitale de la France ?"
    # IDs du prompt dans l'espace du MODÈLE selon la variante : clairs
    # (baseline, partiel sans perm) ou permutés (partiel avec perm, complet).
    ids_clear_csv = ",".join(str(i) for i in tok.encode(SECRET_PROMPT))
    ids_perm_csv = ",".join(str(i) for i in encode(SECRET_PROMPT))

    # Ordre de présentation (priorité utilisateur) : partiel-avec-perm d'abord
    # (la permutation protège le texte), puis les fuites, puis le complet.
    VARIANTS = [
        ("partiel (sans attn, avec perm)", "qwen3-0.6b-partial-perm", ids_perm_csv, "perm"),
        ("baseline (Qwen3-0.6B clair)", "hf:Qwen/Qwen3-0.6B", ids_clear_csv, "clair"),
        ("partiel (sans attn, sans perm)", "qwen3-0.6b-partial-noperm", ids_clear_csv, "clair"),
        ("complet", "qwen3-0.6b-obf", ids_perm_csv, "perm"),
    ]

    def _decode_ids(ids, space):
        # ids récupérés : permutés → codec (unperm + decode) ; clairs → decode direct
        return decode(ids) if space == "perm" else tok.decode(ids)

    _ansi = _re.compile(r"\x1b\[[0-9;]*m")
    attaques_results = []
    for label, ref, ids_csv, space in VARIANTS:
        print(f"== attaque : {label} (model_ref={ref}, canal attn L0, "
              f"{len(ids_csv.split(','))} ids) ==")
        out = !~/modal-venv/bin/modal run modal_app.py::isa_attack --ids $ids_csv --channel attn --layer 0 --steps 300 --model-ref $ref
        # SList.n = sortie CLI joinée par \n (SList.s = espaces : inutilisable ici)
        text = _ansi.sub("", out.n)
        m = _re.search(r"RESULTAT_ISA (\{.*\})", text)
        assert m, f"pas de ligne RESULTAT_ISA dans la sortie pour {label} :\n{text[-2000:]}"
        res = _json.loads(m.group(1))
        rate = res["taux_recuperation_ids_modele"]
        leaked = _decode_ids(res["ids_recuperes"], space)
        attaques_results.append({
            "label": label, "model_ref": ref, "rate": rate,
            "ids_recuperes": res["ids_recuperes"], "texte_fuit": leaked,
        })
        print(f"  taux récupération (ids modèle) : {rate:.1%}")
        print(f"  texte fuit : {leaked!r}")
    print("✓ 5.4 : 4 attaques terminées — taux et textes dans attaques_results")
else:
    print("[RUN_HEAVY=False] attaques ISA sauté — attendu : 4 attaques (attn L0, 300 steps) → taux + texte fuit")

In [ ]:
# 5.5 Tableau comparatif + interprétation — les taux réels proviennent d'un
# run RUN_HEAVY=True (colonne TTRSR) ; en headless la colonne est « — ».
ROWS = [
    ("partiel (sans attn, avec perm)", "claire", "oui",
     "ids permutés récupérés — texte illisible sans clé"),
    ("baseline (Qwen3-0.6B clair)", "claire", "non",
     "fuite en clair"),
    ("partiel (sans attn, sans perm)", "claire", "non",
     "fuite en clair (canal attn)"),
    ("complet", "obfusquée", "oui",
     "défendu (taux faible)"),
]
rates = {r["label"]: r["rate"] for r in attaques_results} \
    if "attaques_results" in globals() else {}

print("| Variante | attn | perm | TTRSR (attn L0) | Résultat |")
print("|---|---|---|---|---|")
for label, attn, perm_, result in ROWS:
    rate = f"{rates[label]:.1%}" if label in rates else "—"
    print(f"| {label} | {attn} | {perm_} | {rate} | {result} |")
print()
print("**Interprétation**")
print("- **La permutation est la défense effective** : sur la variante")
print("  « partiel (sans attn, avec perm) », l'attaque retrouve des ids")
print("  **permutés** — sans la clé (jamais sur le serveur), aucun texte.")
print("- Sans permutation (baseline, partiel sans perm), les ids récupérés")
print("  sont **clairs** : le prompt secret fuit directement via les scores")
print("  d'attention (canal attn, couche 0).")
print("- Le modèle **complet** défend le canal attn : Ẑ (β=8) fait chuter la")
print("  récupération de 27,3 % → 9,1 % (POC grandeur nature, RESULTATS_ISA.md).")
print("- Même avec le canal hidden (95,5 % de récupération sur le POC), les")
print("  ids retrouvés sont permutés : la clé reste le secret ultime.")

### 5.6 Démonstration finale (optionnelle, coûteuse)

Le même arc sur le **vrai `Qwen/Qwen3-8B`**
(`MODEL_FINAL = "Qwen/Qwen3-8B"`) : `transform` ~30-60 min par variante
(CPU Modal), attaque sur A100-40GB (~1,5-2 $/h, quelques minutes par
variante). Reprendre les cellules 5.3-5.4 en remplaçant `Qwen/Qwen3-0.6B`
par `MODEL_FINAL` et les sous-répertoires `qwen3-0.6b-*` par `qwen3-8b-*` —
le modèle de service canonique (`qwen3-8b-obf`, sections 2-3) joue le rôle
de la variante « complet ».

## 6. Matrices clés P̂/Q̂ (h>0) — section réservée

### 6.1 Algorithme 1 — construction P̂/Q̂

Algorithme 1 du papier (arXiv 2603.01499 §5.4) : `P̂` de forme `(d, d+2h)` et
`Q̂` de forme `(d+2h, d)` tels que **P̂·Q̂ = I_d**, avec :
- `d` — dimension du modèle ; `h` — redimensionnement (`h=0` dans le périmètre
  validé par le POC) ; **λ = 0.3** (facteur de bruit sur la matrice `B`) ;
- `B = U + λ·V` (U orthogonale, V gaussienne), `E = E1·E2`, `F = F1·F2` (rang
  plein via produits), `Z` orthogonale sur `d+2h` ; `C` dans le noyau de
  `Fᵀ` et `D` dans le noyau de `E` (annulation croisée → P̂·Q̂ = I).

La démo petite échelle (`d=64, h=8, λ=0.3` → P̂ (64,80) · Q̂ (80,64) = I,
erreur < 1e-10) est déjà exécutée en **1.5** ; l'implémentation **complète**
(h>0, réseau redimensionné en `d+2h`, chaînage inter-couches) est réservée —
référence : spec Secretarius
`docs/superpowers/specs/2026-08-22-aloepri-matrices-cles-design.md`.

### 6.2 À implémenter — h>0, redimensionnement d+2h, chaînage inter-couches

Cellule **stub** : l'exécution du notebook s'arrête à la **Section 5** tant
que la brique `d+2h` n'est pas implémentée. La cellule ci-dessous lève
volontairement `NotImplementedError` (tag nbformat `raises-exception` —
tolérée par l'exécution headless) et sera remplacée par l'implémentation
(spec `docs/superpowers/specs/2026-08-24-aloepri-notebook-design.md` §6).

In [ ]:
# STUB — à implémenter (spec : docs/superpowers/specs/2026-08-24-aloepri-notebook-design.md §6)
# Étapes : (1) redimensionner le réseau en d+2h de bout en bout (embedding,
# couches, lm_head) ; (2) P̂ global unique (conjugaison par couche) ;
# (3) re-mesure ISA hidden L1 — cible TTRSR ≈ 0,82 % (Tableau 4 AloePri).
raise NotImplementedError("Section 6 : matrices clés P̂/Q̂ (h>0) — à implémenter")